### CUDA Performance Benchmarking and Profiling

This notebook demonstrates how to measure, analyze, and profile GPU operations using PyTorch on Google Colab's CUDA runtime.

We perform three primary tasks:
1. **Benchmark Execution (`cuda_benchmark.py`)**: Comparing performance metrics (GFLOPS, Bandwidth, Memory Usage) between CPU and GPU across different tensor operations and dimensions.
2. **Nsight Systems Profiling (`nsys`)**: Capturing low-level GPU kernel timelines, memory copies, and performance counters.
3. **PyTorch Profiler Analysis**: Diving deep into execution trace details, CPU vs. GPU overheads, and operation attribution.

In [1]:
%%writefile cuda_benchmark.py
import time
import json
import statistics
import torch
import torch.nn as nn

def run_cuda_workload(workload_name: str, device_name: str, size: int, warmup: int = 10, runs: int = 40):
    device = torch.device(device_name)
    if device_name == "cuda":
        torch.cuda.reset_peak_memory_stats()

    if workload_name == "MatMul":
        a = torch.randn((size, size), device=device)
        b = torch.randn((size, size), device=device)
        fn = lambda: torch.matmul(a, b)
        flops = 2 * (size ** 3)
        mem_bytes = 3 * (size ** 2) * 4
    elif workload_name == "Elementwise_Add":
        a = torch.randn((size, size), device=device)
        b = torch.randn((size, size), device=device)
        fn = lambda: a + b
        flops = size ** 2
        mem_bytes = 3 * (size ** 2) * 4
    elif workload_name == "MLP_NeuralNet":
        model = nn.Sequential(nn.Linear(size, size), nn.ReLU(), nn.Linear(size, size)).to(device)
        x = torch.randn((256, size), device=device)
        fn = lambda: model(x)
        flops = 2 * 256 * (size ** 2) * 2
        mem_bytes = sum(p.nelement() * 4 for p in model.parameters()) + (x.nelement() * 4)

    # 1. Warm-up runs
    for _ in range(warmup):
        _ = fn()
    if device_name == "cuda":
        torch.cuda.synchronize()

    # 2. Timed runs using Hardware CUDA Events + NVTX Ranges for Nsight Systems
    times_ms = []
    if device_name == "cuda":
        torch.cuda.nvtx.range_push(f"{workload_name}_{size}")
        start_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
        end_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
        for i in range(runs):
            start_events[i].record()
            _ = fn()
            end_events[i].record()
        torch.cuda.synchronize()
        torch.cuda.nvtx.range_pop()
        times_ms = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
        mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = fn()
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000.0)
        mem_mb = mem_bytes / (1024 ** 2)

    median_ms = statistics.median(times_ms)
    gflops = (flops / (median_ms / 1000.0)) / 1e9
    bw_gbs = (mem_bytes / 1e9) / (median_ms / 1000.0)

    print(
        f"{workload_name:<16} | {device_name.upper():<4} | {size:<4}x{size:<4} | "
        f"Med: {median_ms:7.2f} ms | {gflops:8.1f} GFLOPS | BW: {bw_gbs:6.1f} GB/s | Mem: {mem_mb:6.1f} MB"
    )

if __name__ == "__main__":
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print("=" * 98)
    for w in ["MatMul", "Elementwise_Add", "MLP_NeuralNet"]:
        for s in [256, 1024, 4096]:
            for d in ["cpu", "cuda"]:
                run_cuda_workload(w, d, s)
        print("-" * 98)

Writing cuda_benchmark.py


### 1. CUDA Performance Benchmark Script
Above, we wrote `cuda_benchmark.py` to test Matrix Multiplication, Elementwise Addition, and a Multi-Layer Perceptron (MLP) on CPU vs. GPU. It utilizes CUDA Hardware Events to accurately record timings on the GPU without CPU-host synchronization bottlenecks.

In [2]:
!nsys profile --trace=cuda,nvtx --stats=true --force-overwrite=true -o nsight_report python cuda_benchmark.py

GPU Detected: Tesla T4
MatMul           | CPU  | 256 x256  | Med:    0.47 ms |     70.7 GFLOPS | BW:    1.7 GB/s | Mem:    0.8 MB
MatMul           | CUDA | 256 x256  | Med:    0.08 ms |    433.9 GFLOPS | BW:   10.2 GB/s | Mem:    9.1 MB
MatMul           | CPU  | 1024x1024 | Med:   17.38 ms |    123.6 GFLOPS | BW:    0.7 GB/s | Mem:   12.0 MB
MatMul           | CUDA | 1024x1024 | Med:    0.83 ms |   2583.6 GFLOPS | BW:   15.1 GB/s | Mem:   24.1 MB
MatMul           | CPU  | 4096x4096 | Med: 1153.87 ms |    119.1 GFLOPS | BW:    0.2 GB/s | Mem:  192.0 MB
MatMul           | CUDA | 4096x4096 | Med:   35.27 ms |   3897.2 GFLOPS | BW:    5.7 GB/s | Mem:  264.1 MB
--------------------------------------------------------------------------------------------------
Elementwise_Add  | CPU  | 256 x256  | Med:    0.02 ms |      3.8 GFLOPS | BW:   45.4 GB/s | Mem:    0.8 MB
Elementwise_Add  | CUDA | 256 x256  | Med:    0.04 ms |      1.8 GFLOPS | BW:   22.1 GB/s | Mem:    9.1 MB
Elementwise_Add  | CPU

### 2. Nsight Systems Profiling
Using `nsys profile`, we executed the benchmark while recording CUDA API calls and NVTX user-defined ranges. This provides a deep performance profile (`nsight_report.nsys-rep`) detailing GPU utilization, memory copy overhead, and specific kernel execution durations.

In [3]:
import torch
from torch.profiler import profile, ProfilerActivity

a = torch.randn((4096, 4096), device="cuda")
b = torch.randn((4096, 4096), device="cuda")
_ = torch.matmul(a, b) # Warm-up
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    with torch.profiler.record_function("MatMul_4096"):
        _ = torch.matmul(a, b)
    with torch.profiler.record_function("Elementwise_Add_4096"):
        _ = a + b
    torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            MatMul_4096         0.72%     388.082us         5.72%       3.078ms       3.078ms       0.000us         0.00%      94.990ms      94.990ms             1  
                                           aten::matmul         0.04%      19.076us         5.00%       2.690ms       2.690ms       0.000us         0.00%      94.990ms      94.990ms             1  
         

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


### 3. PyTorch Profiler Integration
The PyTorch Profiler run above isolates the 4096x4096 `MatMul` and `Elementwise_Add` operators, showcasing `Self CPU` and `Self CUDA` runtimes. This tabular view helps trace exactly which underlying CUDA kernels (e.g., `sgemm`) were invoked and how much time was spent on memory operations or kernel launch overheads.